# JopPyLib Notebook
A notebook interface to work with JopPyLib. JopPyLib wraps the [Joplin Data API](https://joplinapp.org/help/api/references/rest_api/) to allow easy manipulation of your notes from Python. This notebook will focus on functionality. Documentation can be found on [JopPyLib Github](https://github.com/jeroenkroesen/joppylib).

## Get started

To use this notebook, you need the following:
* JopPyLib installed (I recommend in a virtual env)
* Joplin running on the same machine, with the Web Clipper service enabled
* Authentication to your Joplin client. This can be handled in two ways:
    * Manual authentication: copy the api-key from Joplin and paste it into the instantiation of the JopPyLib clients
    * Interactive: On instantiation, the Higher-level client will request the API key from Joplin. Joplin will display a pop-up for you to authorize.  
  
There are 2 interfaces to Joplin in JopPyLib:   
  
### High-level API
The High-level API requires only settings and will take care of everything else
```python
from joppylib import default_settings, JoplinClient
joplin = JoplinClient(default_settings)

# Get all notes
all_my_notes = joplin.note.get_multi()
```

### Low-level API
The low-level API has a rather functional style and requires passing settings and the API-key on each call, but is more modular and flexible than the high-level API.
```python
from joppylib import config, connection, api_client
settings = config.Settings()
joplin_api = api_client.APIClient()

# Authenticate
if connection.check_connection(settings.base_url, settings.ping_route):
    print('Joplin API online')
    result = connection.get_auth_token(settings.base_url, settings.auth_init_route, settings.auth_check_route)
    if result['status'] == 'accepted':
        api_key = result['token']
        print('API-key received')
    else: 
        print('Authorization failed')
else:
    print('Cannot connect to Joplin API')

# Get all notes
all_my_notes = joplin_api.note.get_multi(api_key, settings)
```

More documentation will follow on Github. For now, after instantion run:
```python
help(joplin) # high-level
help(joplin_api) # low-level
```

## Imports and basic setup

In [1]:
# Visual aids
import rich
from rich import print, inspect, print_json # Fancy datastructure printing

# Default stuff from Python
#import json # Structured data yay
#from pathlib import Path # Find your way around filesystems
#import requests # API stuff

# Are you my type?
from typing import Dict, List, Optional, Any
from collections.abc import Callable

# Lower level API import
from joppylib import config, connection, api_client
joplin_api = api_client.APIClient()

# Higherlevel API import
from joppylib import default_settings, JoplinClient

***

# High-level API

## Setup and connection

In [2]:
joplin = JoplinClient(default_settings)

***

## Setup for testing

In [37]:
# Test if notebook 'z.Test' exists and if not create it
test_folder_title = 'z.Test'
query = test_folder_title
test_folder_search_result = joplin.folder.search(query)

if test_folder_search_result['data'] and test_folder_search_result['data'][0]['title'] == test_folder_title:
    test_folder_id = test_folder_search_result['data'][0]['id']
    print(f'found folder {test_folder_title} with id: {test_folder_id}')
else:
    print('No test folder found, creating')
    data = {
        'parent_id': '',
        'title': test_folder_title
    }
    created_test_folder = joplin.folder.create(data).json()
    test_folder_id = created_test_folder['id']
    print(f'Created folder {test_folder_title} with id: {test_folder_id}')


# Test if tag '...test' exists and if not create it
test_tag_title = '...test'
query = test_tag_title
test_tag_search_result = joplin.tag.search(query)

if test_tag_search_result['data'] and test_tag_search_result['data'][0]['title'] == test_tag_title:
    test_tag_id = test_tag_search_result['data'][0]['id']
    print(f'found tag {test_tag_title} with id: {test_folder_id}')
else:
    print('No test tag found, creating')
    data = {
        'parent_id': '',
        'title': test_tag_title
    }
    created_test_tag = joplin.tag.create(data).json()
    test_tag_id = created_test_tag['id']
    print(f'Created tag {test_tag_title} with id: {test_tag_id}')

found folder z.Test with id: 1d12bb9e04314eb395452ae0e7b6e415

No test tag found, creating

Created tag ...test with id: 57ed05ed70e94429a7fc586fc547ed8f

***

## Notes

### Create

In [ ]:
## Happy flow assumed

# Set folder for the note:
note_data_create = {
    'title': '..001 TestNote',
    'body': '## Dude this is a cool note',
    'parent_id': test_folder_id 
}
created_note = joplin.note.create(note_data_create).json()
created_note_id = created_note['id']
print_json(data=created_note)

### Search

In [ ]:
query = 'title:..001 TestNote'
fields = ['id', 'title', 'body']
order_by = 'title'
order_dir = 'ASC' # or 'DESC'
debug = True

search_results_note = joplin.note.search(
    query, fields, order_by, order_dir, debug
)
print(search_results_note)

### Get

In [ ]:
# Get all notes
fields = ['id', 'title']
order_by = 'title'
order_dir = 'ASC' # or 'DESC'
debug = True

all_notes = joplin.note.get_multi(
    fields, order_by, order_dir, debug
)
json_print(data=all_notes)

In [ ]:
# Get note by ID
fields = ['id', 'title', 'body']
gotten_note = joplin.note.get(created_note_id, fields=fields).json()
print_json(data=gotten_note)

### Show all tags

In [6]:
# Show all tags of this note
created_note_all_tags = joplin.note.get_all_tags(created_note_id)
print_json(data=created_note_all_tags)

### Update

In [ ]:
# Update test note
note_data_update = {
    'title': '..001 TestNote AND NOW THE TITLE IS CHANGED',
    'body': '## Dude this is a cool note. I KNOW RIGHT!' 
}
updated_note = joplin.note.update(created_note_id, note_data_update).json()
print_json(data=updated_note)

### Delete

In [ ]:
# Delete test note
deleted_note = joplin.note.delete(created_note_id)
print(deleted_note)

In [ ]:
# Delete test note permanently (remove from trash)
deleted_note = joplin.note.delete(created_note_id, trash=False)
print(deleted_note)

***

## Tags

### Search

In [ ]:
test_tag_title = '...test'
query = test_tag_title
test_tag_search_result = joplin.tag.search(query)
print(test_tag_search_result)

### Create

In [ ]:
# Create a tag named ...test
# test_tag_title = '...test' # Also defined in testing cell above
data = {
    'parent_id': '',
    'title': test_tag_title
}
created_test_tag = joplin.tag.create(data).json()
test_tag_id = created_test_tag['id']
print_json(data=created_test_tag)

### Add to note

In [ ]:
# Tag the note
note_tagged = joplin.tag.add_to_note(test_tag_id, created_note_id).json()
print_json(data=note_tagged)

### Remove from note

In [ ]:
# Remove tag from note
note_untagged = joplin.tag.remove_from_note(test_tag_id, created_note_id)
print(note_untagged)

### Get

In [ ]:
# Get all
all_tags = joplin.tag.get_multi()
print_json(data=all_tags)

In [ ]:
# Get one
get_test_tag = joplin.tag.get(test_tag_id).json()
print_json(data=get_test_tag)

### Update

In [ ]:
update_tag_data = {'title': '...test2'}
updated_tag = joplin.tag.update(test_tag_id, update_tag_data).json()
print_json(data=updated_tag)

### Delete

In [35]:
deleted_tag_resp = joplin.tag.delete(test_tag_id) # No response body.. Headers inspectable if you like

***

## Folders

### Create

In [ ]:
# Todo

### Search

In [ ]:
# Todo

### Get

In [ ]:
# Todo

### Update

In [ ]:
# Todo

### Delete

In [ ]:
# Todo

***

## Revisions

### Create

In [ ]:
# Todo

### Search

In [ ]:
# Todo

### Get

In [ ]:
# Todo

### Update

In [ ]:
# Todo

### Delete

In [ ]:
# Todo

***

## Resources

### Create

In [ ]:
# Todo

### Search

In [ ]:
# Todo

### Get

In [ ]:
# Todo

### Update

In [ ]:
# Todo

### Delete

In [ ]:
# Todo

***

# Low-level API

## Setup and connection

In [ ]:
# Setup
settings = config.Settings()
print(settings)

if connection.check_connection(settings.base_url, settings.ping_route):
    print('Joplin API online')
    result = connection.get_auth_token(settings.base_url, settings.auth_init_route, settings.auth_check_route)
    if result['status'] == 'accepted':
        api_key = result['token']
        print('API-key received')
    else: 
        print('Authorization failed')
else:
    print('Cannot connect to Joplin API')



***

## Creating

In [ ]:
## Notebook
notebook_data = {
    'title': 'z.Test'
}
created_notebook = joplin_api.notebook.create(api_key, settings, notebook_data)
print(created_notebook)

In [ ]:
# POST calls
## Note
note_data = {
    'title': '..001 TestNote',
    'body': '## Dude this is a cool note',
    'parent_id': '6f50f25b18e944438c17476d4be9c14c' # ID of 01.Mind
}
created_note = joplin_api.note.create(api_key, settings, note_data)
print(created_note)

In [ ]:
## Tag
tag_data = {'title': '...test'}
created_tag = joplin_api.tag.create(api_key, settings, tag_data)
print(created_tag)

***

## Searching

In [ ]:
# Notes:
fields_for_search = ['id', 'title'] # Fields to return
my_query = 'title:2025-02-25 Tuesday Diary' # Search query
search_result_test_debug = joplin_api.note.search(api_key, settings, query=my_query, fields=fields_for_search, debug=True)
search_result_test = joplin_api.note.search(api_key, settings, query=my_query, fields=fields_for_search)

print_json(data=search_result_test) # When debug=False (default)
inspect(search_result_test_debug) # When debug=True

In [3]:
# notebooks
fieldssearch = ['id', 'title']
mind_nb_query = '1.Mind'
search_result_test = joplin_api.notebook.search(api_key, settings, query=mind_nb_query, fields=fieldssearch)

In [3]:
# tags
fieldssearch = ['id', 'title']
tag_query = 'time.2025'
search_result_test = joplin_api.tag.search(api_key, settings, query=tag_query, fields=fieldssearch)

In [ ]:
print(search_result_test)

***

## Getting

In [ ]:
# Notes:
fields_for_multi_get = ['id', 'title'] # Fields to return
get_multi_result_test_debug = joplin_api.note.get_multi(api_key, settings, fields=fields_for_multi_get, debug=True)
get_multi_result_test = joplin_api.note.get_multi(api_key, settings, fields=fields_for_multi_get)

print_json(data=get_multi_result_test) # When debug=False (default)
inspect(get_multi_result_test_debug) # When debug=True

In [ ]:
# All tags for a note
note_id = '5d584dd4863f49349b966872fddef3b5'
tags_for_note = joplin_api.note.get_all_tags(
    api_key,
    settings,
    note_id
)

print_json(data=tags_for_note)

In [ ]:
# Tags
fields_for_multi_get = ['id', 'title'] # Fields to return
get_multi_result_test_debug = joplin_api.tag.get_multi(api_key, settings, fields=fields_for_multi_get, debug=True)
get_multi_result_test = joplin_api.tag.get_multi(api_key, settings, fields=fields_for_multi_get)

print_json(data=get_multi_result_test) # When debug=False (default)
inspect(get_multi_result_test_debug) # When debug=True

In [ ]:
# Notebooks
fields_for_multi_get = ['id', 'title'] # Fields to return
get_multi_result_test_debug = joplin_api.notebook.get_multi(api_key, settings, fields=fields_for_multi_get, debug=True)
get_multi_result_test = joplin_api.notebook.get_multi(api_key, settings, fields=fields_for_multi_get)

print_json(data=get_multi_result_test) # When debug=False (default)
inspect(get_multi_result_test_debug) # When debug=True

***